In [2]:
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_agentchat.teams import SelectorGroupChat
from autogen_agentchat.agents import AssistantAgent, UserProxyAgent
from autogen_agentchat.ui import Console
from autogen_agentchat.conditions import TextMentionTermination
import sys
import os
sys.path.append(os.path.abspath(".."))
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
from langchain_community.tools.tavily_search import TavilySearchResults

C:\Users\davis\AppData\Local\Temp\ipykernel_10580\723937994.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools.tavily_search import TavilySearchResults


In [4]:
model_client = OpenAIChatCompletionClient(
    model='gpt-3.5-turbo'
)

In [5]:
async def research_tool(query: str) -> str:
    """Search the web for information"""
    search = TavilySearchResults(max_results=3)
    results = search.invoke(query)
    return str(results)

In [ ]:
supervisor_agent = AssistantAgent(
    "SupervisorAgent",
    description="An agent for planning and supervising which agent to act next",
    model_client=model_client,
    system_message="""You are a supervisor AI. Decide which agent should act next
            Options:
            - research_agent → if more information is needed.
            - writer → if a summary or refinement is needed.
            - end → if the task is complete
            Respond ONLY with one of the following: research_agent, writer_agent, or end."""
)

In [8]:
research_agent = AssistantAgent(
    "ResearchAgent",
    description="An agent for researching about the given topic using the tool provided",
    model_client=model_client,
    tools= [research_tool],
    system_message="""
    You are a researcher agent, and once you are given a topic to research about, you should make use of tool provided to you for the research purpose.
    You should only be researching based on the tool provided to you
    """,
)

In [9]:
writer_agent = AssistantAgent(
    "WriterAgent",
    description="An agent for summarizing the research",
    model_client=model_client,
    system_message="""
    You will be given the research results, and your task is only the summarize them in a professional manner within 150 words
    """,
)